# Sistema de Telemetria Espacial

**Atividade Integradora - Ciencia da Computacao**

Este notebook implementa uma simulacao didatica de verificacao de telemetria para decidir entre **PRONTO PARA DECOLAR** e **DECOLAGEM ABORTADA**.

> As faixas de seguranca utilizadas sao parametros didaticos definidos para a atividade e nao representam especificacoes operacionais de uma nave real.

## 1. Dados e limites de seguranca

Sao avaliados temperatura interna e externa, integridade estrutural, energia, pressao dos tanques e status dos modulos criticos.

In [1]:
# Sistema didatico de verificacao de telemetria espacial

LIMITES = {
    "temperatura_interna": (18, 27),
    "temperatura_externa": (-60, 40),
    "energia_minima": 80,
    "pressao_tanques": (220, 260),
}

def verificar_telemetria(dados):
    erros = []

    temp_min, temp_max = LIMITES["temperatura_interna"]
    if not temp_min <= dados["temperatura_interna"] <= temp_max:
        erros.append("Temperatura interna fora da faixa segura.")

    ext_min, ext_max = LIMITES["temperatura_externa"]
    if not ext_min <= dados["temperatura_externa"] <= ext_max:
        erros.append("Temperatura externa fora da faixa segura.")

    if dados["integridade_estrutural"] != 1:
        erros.append("Falha na integridade estrutural.")

    if dados["energia"] < LIMITES["energia_minima"]:
        erros.append("Nivel de energia insuficiente.")

    p_min, p_max = LIMITES["pressao_tanques"]
    if not p_min <= dados["pressao_tanques"] <= p_max:
        erros.append("Pressao dos tanques fora da faixa segura.")

    for modulo in ["navegacao", "comunicacao", "propulsao"]:
        if dados[modulo] != 1:
            erros.append(f"Falha no modulo de {modulo}.")

    status = "PRONTO PARA DECOLAR" if not erros else "DECOLAGEM ABORTADA"
    return status, erros


## 2. Cenario 1 - Condicoes seguras

In [2]:
telemetria = {
    "temperatura_interna": 22,
    "temperatura_externa": -35,
    "integridade_estrutural": 1,
    "energia": 92,
    "pressao_tanques": 245,
    "navegacao": 1,
    "comunicacao": 1,
    "propulsao": 1,
}

status, erros = verificar_telemetria(telemetria)
print("=== ANALISE DE TELEMETRIA ===")
print("Resultado:", status)
if erros:
    for erro in erros:
        print("-", erro)

=== ANALISE DE TELEMETRIA ===
Resultado: PRONTO PARA DECOLAR


## 3. Cenario 2 - Teste de deteccao de anomalias

In [3]:
telemetria_anomala = {
    "temperatura_interna": 30,
    "temperatura_externa": -35,
    "integridade_estrutural": 1,
    "energia": 62,
    "pressao_tanques": 275,
    "navegacao": 1,
    "comunicacao": 1,
    "propulsao": 0,
}

status, erros = verificar_telemetria(telemetria_anomala)
print("=== ANALISE DE TELEMETRIA ===")
print("Resultado:", status)
for erro in erros:
    print("-", erro)

=== ANALISE DE TELEMETRIA ===
Resultado: DECOLAGEM ABORTADA
- Temperatura interna fora da faixa segura.
- Nivel de energia insuficiente.
- Pressao dos tanques fora da faixa segura.
- Falha no modulo de propulsao.


## 4. Analise energetica

A simulacao considera 500 kWh de capacidade total, 92% de carga, 250 kWh de consumo estimado durante a decolagem e 8% de perdas energeticas.

In [4]:
def analisar_energia(capacidade_total_kwh, carga_atual_pct, consumo_decolagem_kwh, perdas_pct):
    energia_carregada = capacidade_total_kwh * (carga_atual_pct / 100)
    perdas_kwh = energia_carregada * (perdas_pct / 100)
    energia_util = energia_carregada - perdas_kwh
    energia_restante = energia_util - consumo_decolagem_kwh
    percentual_restante = (energia_restante / capacidade_total_kwh) * 100

    return {
        "energia_carregada_kwh": energia_carregada,
        "perdas_kwh": perdas_kwh,
        "energia_util_kwh": energia_util,
        "energia_restante_kwh": energia_restante,
        "percentual_restante": percentual_restante,
    }


In [5]:
resultado_energia = analisar_energia(
    capacidade_total_kwh=500,
    carga_atual_pct=92,
    consumo_decolagem_kwh=250,
    perdas_pct=8,
)

for chave, valor in resultado_energia.items():
    print(f"{chave}: {valor:.2f}")

energia_carregada_kwh: 460.00
perdas_kwh: 36.80
energia_util_kwh: 423.20
energia_restante_kwh: 173.20
percentual_restante: 34.64


## 5. Analise assistida por IA

### Prompt utilizado

> Classifique os dados de telemetria apresentados, identifique anomalias e descreva os principais riscos para a decolagem. Considere como anomalia qualquer valor fora das faixas de seguranca definidas no projeto.

### Sintese da analise

- **Cenario seguro:** parametros dentro dos limites e modulos criticos operacionais; classificacao de baixo risco.
- **Cenario anomalo:** temperatura interna elevada, energia insuficiente, sobrepressao e falha de propulsao; classificacao de alto risco e decolagem deve ser abortada.
- **Risco residual:** mesmo com valores normais, os dados devem continuar sendo monitorados porque sensores podem falhar e as condicoes podem mudar durante a operacao.

## 6. Conclusao

A integracao entre dados de sensores, regras de decisao e analise computacional permite identificar condicoes inseguras antes da decolagem. O algoritmo prioriza seguranca: qualquer falha critica e suficiente para interromper a operacao.